<a href="https://colab.research.google.com/github/vineet-crypto/Recommendation-Systems-for-Personalized-Content-Discovery/blob/main/cult_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas numpy matplotlib scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.1 MB/s eta 0:00:00


In [7]:
import pandas as pd
import numpy as np
import time
from collections import defaultdict
from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import train_test_split

# =================
# 1. CONFIGURATION
# =================
FILE_PATH = 'combined_data_1.txt'
MOVIE_META_PATH = 'movie_titles.csv'
OUTPUT_FILE = 'final_user_recommendations.csv'

TOTAL_NETFLIX_RATINGS = 100480507
TOTAL_NETFLIX_MOVIES = 17770

LOAD_FULL_FILE = False   # Set to True to process the entire ~24M rows of fraction 1
USER_EXPORT_LIMIT = 499  # Number of existing users to export

# ==========================
# 2. OPTIMIZED DATA LOADING
# ==========================
print(f"Starting data ingestion from {FILE_PATH}...")

def load_fractional_data(file_path, load_all=True, max_rows=2000000):
    data = {'User_ID': [], 'Rating': [], 'Movie_ID': []}
    current_movie = None
    count = 0

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue

            if line.endswith(':'):
                try:
                    current_movie = int(line[:-1])
                except ValueError:
                    continue
            else:
                parts = line.split(',')
                if len(parts) < 3:
                    continue

                try:
                    user_id = int(parts[0])
                    rating = float(parts[1])

                    data['User_ID'].append(user_id)
                    data['Rating'].append(rating)
                    data['Movie_ID'].append(current_movie)
                    count += 1
                except ValueError:
                    continue

                if not load_all and count >= max_rows:
                    break

    return pd.DataFrame(data)

df = load_fractional_data(FILE_PATH, load_all=LOAD_FULL_FILE)
movies_df = pd.read_csv(MOVIE_META_PATH, encoding='ISO-8859-1', header=None,
                        names=['Movie_ID', 'Year', 'Title'], on_bad_lines='skip')
movie_title_dict = dict(zip(movies_df['Movie_ID'], movies_df['Title']))

print(f"Data loaded successfully.")

# =============================
# 3. EXPLORATORY DATA ANALYSIS
# =============================
print("\n" + "="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

actual_ratings = len(df)
actual_movies = df['Movie_ID'].nunique()
unique_users = df['User_ID'].nunique()

print(f"Total Logged Ratings in Sample: {actual_ratings:,}")
print(f"Unique Movies Found: {actual_movies:,}")
print(f"Unique Users Found: {unique_users:,}")

# A. User Activity Patterns
user_activity = df.groupby('User_ID').size()
print(f"Average ratings per user: {user_activity.mean():.2f}")

# B. Content Popularity Trends
movie_popularity = df.groupby('Movie_ID').size()
print(f"Average ratings per movie: {movie_popularity.mean():.2f}")

# C. Rating Distributions
rating_dist = df['Rating'].value_counts().sort_index()
print("\nRating Distribution:")
for rating, r_count in rating_dist.items():
    print(f"  {rating} Stars: {r_count:,} ({r_count/len(df)*100:.1f}%)")

# D. Data Sparsity Matrix
total_possible_interactions = unique_users * actual_movies
sparsity = 1.0 - (actual_ratings / total_possible_interactions)
print(f"\nCalculated Data Sparsity Matrix: {sparsity * 100:.4f}%")

# ============================================================
# 4. ADVANCED COLD START PRE-COMPUTATION (NEW USERS STRATEGY)
# ============================================================
print("\nPre-computing high-quality popularity fallback matrices for Cold Start scenarios...")
movie_stats = df.groupby('Movie_ID').agg(
    avg_rating=('Rating', 'mean'),
    rating_count=('Rating', 'count')
)

min_count_threshold = movie_stats['rating_count'].median()
qualified_movies = movie_stats[movie_stats['rating_count'] >= min_count_threshold]
cold_start_top_10 = qualified_movies.sort_values(by='avg_rating', ascending=False).head(10)

# ================================
# 5. TRAIN-TEST SPLIT METHODOLOGY
# ================================
print("\nPreparing train-test split (80% Train / 20% Test)...")
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['User_ID', 'Movie_ID', 'Rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# ===============================
# 6. EVALUATION METRICS (MAP@10)
# ===============================
def calculate_map_at_10(predictions, relevance_threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = []
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((true_r >= relevance_threshold) for (_, true_r) in user_ratings)

        if n_rel == 0: continue

        hits, sum_precs = 0, 0.0
        for i, (est, true_r) in enumerate(user_ratings[:10]):
            if true_r >= relevance_threshold:
                hits += 1
                sum_precs += hits / (i + 1.0)

        precisions.append(sum_precs / min(n_rel, 10))
    return np.mean(precisions) if precisions else 0.0

# ==============================================
# 7. MODEL DEVELOPMENT & ACCELERATED COMPARISON
# ==============================================
print("\n" + "="*60)
print("MODEL TRAINING & METRIC COMPARISON")
print("="*60)

print("Training Model 1: SVD (Matrix Factorization)...")
svd_model = SVD(n_factors=15, n_epochs=10, random_state=42)
svd_model.fit(trainset)
preds_svd = svd_model.test(testset)
rmse_svd = accuracy.rmse(preds_svd, verbose=False)
map10_svd = calculate_map_at_10(preds_svd)

print("Training Model 2: Item-Based Collaborative Filtering...")
sim_options = {'name': 'cosine', 'user_based': False}
knn_model = KNNBasic(sim_options=sim_options, verbose=False)
knn_model.fit(trainset)
preds_knn = knn_model.test(testset)
rmse_knn = accuracy.rmse(preds_knn, verbose=False)
map10_knn = calculate_map_at_10(preds_knn)

print(f"\nSVD Matrix Factorization -> Validation RMSE: {rmse_svd:.4f} | Validation MAP@10: {map10_svd:.4f}")
print(f"Item-Based KNN -> Validation RMSE: {rmse_knn:.4f} | Validation MAP@10: {map10_knn:.4f}")

# ==========================================================
# 8. GENERATING PREDICTIONS WITH COLD START INTEGRATION
# ==========================================================
print("\n" + "="*60)
print("GENERATING PERSONALIZED PREDICTIONS WITH COLD-START FALLBACK")
print("="*60)
print(f"Computing Top-10 lists and saving outputs to: '{OUTPUT_FILE}'...")

best_architecture = SVD(n_factors=15, n_epochs=10, random_state=42) if rmse_svd < rmse_knn else knn_model
full_trainset = data.build_full_trainset()
best_architecture.fit(full_trainset)

# Setup our target user base array
existing_users = list(df['User_ID'].unique()[:USER_EXPORT_LIMIT])

# Inject a single clean test entry for a brand new cold-start user
mock_cold_start_users = ['NEW_USER_999']
target_user_pool = mock_cold_start_users + existing_users

print(f"Total users to process: {len(target_user_pool)} (1 New User + {USER_EXPORT_LIMIT} Existing Users)")

all_movies = set(df['Movie_ID'].unique())
user_viewing_histories = df.groupby('User_ID')['Movie_ID'].apply(set).to_dict()

with open(OUTPUT_FILE, 'w', encoding='utf-8') as output_stream:
    output_stream.write("User_ID,Recommendation_Rank,Movie_ID,Predicted_Rating,Movie_Title,Strategy_Used\n")

    for processed_count, user_id in enumerate(target_user_pool, 1):
        top_10_picks = []
        strategy = "Personalized (SVD)"

        try:
            if isinstance(user_id, str) and user_id.startswith('NEW_USER'):
                is_cold_start = True
            else:
                _ = best_architecture.trainset.to_inner_uid(user_id)
                is_cold_start = False
        except ValueError:
            is_cold_start = True

        if is_cold_start:
            strategy = "Cold-Start Fallback (Popular/High-Quality)"
            for rank, (movie_id, row) in enumerate(cold_start_top_10.iterrows(), 1):
                top_10_picks.append((movie_id, row['avg_rating']))
        else:
            seen = user_viewing_histories.get(user_id, set())
            unseen_movies = all_movies - seen

            user_scores = []
            for movie_id in unseen_movies:
                prediction = best_architecture.predict(user_id, movie_id)
                user_scores.append((movie_id, prediction.est))

            user_scores.sort(key=lambda x: x[1], reverse=True)
            top_10_picks = user_scores[:10]

        for rank, (movie_id, score) in enumerate(top_10_picks, 1):
            clean_title = movie_title_dict.get(movie_id, "Unknown Title").replace(",", "")
            output_stream.write(f"{user_id},{rank},{movie_id},{score:.3f},{clean_title},{strategy}\n")

        if processed_count % 100 == 0 or processed_count == len(target_user_pool):
            print(f" Successfully exported recommendations for {processed_count}/{len(target_user_pool)} users.")

print(f"\nPipeline successfully completed! Check '{OUTPUT_FILE}' to view personalized vs cold-start listings.")

Starting data ingestion from combined_data_1.txt...
Data loaded successfully.

EXPLORATORY DATA ANALYSIS
Total Logged Ratings in Sample: 2,000,000
Unique Movies Found: 361
Unique Users Found: 342,445
Average ratings per user: 5.84
Average ratings per movie: 5540.17

Rating Distribution:
  1.0 Stars: 89,009 (4.5%)
  2.0 Stars: 192,943 (9.6%)
  3.0 Stars: 559,630 (28.0%)
  4.0 Stars: 697,744 (34.9%)
  5.0 Stars: 460,674 (23.0%)

Calculated Data Sparsity Matrix: 98.3822%

Pre-computing high-quality popularity fallback matrices for Cold Start scenarios...

Preparing train-test split (80% Train / 20% Test)...

MODEL TRAINING & METRIC COMPARISON
Training Model 1: SVD (Matrix Factorization)...
Training Model 2: Item-Based Collaborative Filtering...

SVD Matrix Factorization -> Validation RMSE: 0.9852 | Validation MAP@10: 0.9209
Item-Based KNN -> Validation RMSE: 1.0906 | Validation MAP@10: 0.9037

GENERATING PERSONALIZED PREDICTIONS WITH COLD-START FALLBACK
Computing Top-10 lists and saving o